# C4 데이터셋의 비영어 텍스트 분석

C4(Colossal Clean Crawled Corpus)는 `langdetect`를 사용해 영어일 확률이 0.99 미만인 텍스트를 필터링한 영어 데이터셋입니다.  
그러나 여전히 비영어 텍스트가 다수 포함되어 있습니다.

이 노트북에서는 C4의 `realnewslike` 서브셋에서 비영어(한국어) 텍스트가 등장하는 인스턴스를 찾고,  
어떤 맥락에서 등장하는지 분석합니다.

## 1. 라이브러리 설치

In [1]:
!pip install datasets langdetect langid -q

## 2. C4 realnewslike 서브셋 로드

In [2]:
from datasets import load_dataset

# streaming=True로 전체 다운로드 없이 샘플링
dataset = load_dataset(
    "allenai/c4",
    "realnewslike",
    split="train",
    streaming=True,
    trust_remote_code=True,
)

print("C4 realnewslike 스트리밍 로드 완료")

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/512 [00:00<?, ?it/s]

C4 realnewslike 스트리밍 로드 완료


## 3. 언어 감지 함수 정의

In [4]:
import re
from langdetect import detect_langs
from langdetect.lang_detect_exception import LangDetectException


def detect_language(text: str) -> tuple[str, float]:
    """텍스트의 언어와 확률을 반환합니다."""
    try:
        langs = detect_langs(text)
        top = langs[0]
        return top.lang, top.prob
    except LangDetectException:
        return "unknown", 0.0


def contains_non_ascii(text: str) -> bool:
    """ASCII 범위를 벗어나는 문자가 포함되어 있는지 확인합니다."""
    return bool(re.search(r'[^\x00-\x7F]', text))


def extract_non_english_sentences(text: str, target_lang: str = "ko") -> list[str]:
    """텍스트에서 문장 단위로 분리해 비영어 문장을 반환합니다."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    non_english = []
    for sent in sentences:
        if len(sent.strip()) < 10:
            continue
        lang, prob = detect_language(sent)
        if lang == target_lang and prob > 0.8:
            non_english.append(sent.strip())
    return non_english

## 4. 비영어 텍스트 인스턴스 탐색

C4 realnewslike 서브셋에서 샘플을 스캔해 비영어(한국어) 텍스트가 포함된 문서를 찾습니다.

In [5]:
TARGET_LANG = "ko"   # 찾을 언어: 한국어
SCAN_LIMIT  = 50_000 # 스캔할 최대 문서 수
RESULT_LIMIT = 20    # 수집할 최대 결과 수

found_examples = []

for i, sample in enumerate(dataset):
    if i >= SCAN_LIMIT or len(found_examples) >= RESULT_LIMIT:
        break

    text = sample["text"]

    # 빠른 사전 필터: non-ASCII 문자가 없으면 한국어 포함 불가
    if not contains_non_ascii(text):
        continue

    doc_lang, doc_prob = detect_language(text)
    non_en_sents = extract_non_english_sentences(text, target_lang=TARGET_LANG)

    if non_en_sents:
        found_examples.append({
            "index": i,
            "url": sample.get("url", ""),
            "doc_lang": doc_lang,
            "doc_lang_prob": round(doc_prob, 4),
            "text_snippet": text[:500],   # 앞부분 500자
            "non_english_sentences": non_en_sents,
            "full_text": text,
        })
        print(f"[{len(found_examples):02d}] 문서 #{i} | URL: {sample.get('url','')[:60]}")
        print(f"     문서 언어: {doc_lang} (확률 {doc_prob:.4f})")
        print(f"     비영어 문장 수: {len(non_en_sents)}")
        print()

print(f"\n탐색 완료 | 스캔 문서: {i+1}개 | 비영어 문서 발견: {len(found_examples)}개")

[01] 문서 #6391 | URL: https://searchworks.stanford.edu/?f%5Bauthor_other_facet%5D%
     문서 언어: en (확률 1.0000)
     비영어 문장 수: 1



KeyboardInterrupt: 

## 5. 발견된 비영어 텍스트 맥락 분석

In [6]:
def find_context(full_text: str, target_sentence: str, window: int = 200) -> str:
    """비영어 문장 전후 window 글자를 반환합니다."""
    idx = full_text.find(target_sentence)
    if idx == -1:
        return "(문장 위치를 찾을 수 없음)"
    start = max(0, idx - window)
    end   = min(len(full_text), idx + len(target_sentence) + window)
    before = full_text[start:idx]
    after  = full_text[idx + len(target_sentence):end]
    return f"...{before}[[ {target_sentence} ]]{after}..."


for ex in found_examples[:5]:   # 상위 5개만 출력
    print("=" * 80)
    print(f"문서 인덱스 : {ex['index']}")
    print(f"URL         : {ex['url']}")
    print(f"문서 감지 언어: {ex['doc_lang']} (확률 {ex['doc_lang_prob']})")
    print()
    for j, sent in enumerate(ex["non_english_sentences"][:3], 1):  # 문장 최대 3개
        print(f"  [비영어 문장 {j}] {sent}")
        print(f"  [맥락]")
        print(f"  {find_context(ex['full_text'], sent, window=150)}")
        print()
    print()

문서 인덱스 : 6391
URL         : https://searchworks.stanford.edu/?f%5Bauthor_other_facet%5D%5B%5D=Shanghai+dian+ying+zhi+pian+chang&f%5Blanguage%5D%5B%5D=Chinese
문서 감지 언어: en (확률 1.0)

  [비영어 문장 1] 北京 : 北京电视艺术中心音像出版社, 2007.
  [맥락]
  ...ter to the historical trend and the will of the Chinese nation."--Container.
Beijing : Beijing dian shi yi shu zhong xin yin xiang chu ban she, 2007. [[ 北京 : 北京电视艺术中心音像出版社, 2007. ]]
Video — 1 videodisc (132 min.) : sd., col. ; 4 3/4 in.
Yi ge xiao nu hai can jia "quan jia zong dong yuan" bi san, zui zhong zai ge shi ge yang de re...




## 6. 통계 요약

In [7]:
import collections

# 문서당 비영어 문장 수 분포
sent_counts = [len(ex["non_english_sentences"]) for ex in found_examples]
doc_langs   = collections.Counter(ex["doc_lang"] for ex in found_examples)

print(f"발견된 문서 수          : {len(found_examples)}")
print(f"문서당 비영어 문장 수   : 평균 {sum(sent_counts)/max(len(sent_counts),1):.2f} | 최대 {max(sent_counts, default=0)}")
print(f"문서 수준 감지 언어 분포: {dict(doc_langs)}")

# 비영어 문장이 전체 텍스트에서 차지하는 비율
ratios = []
for ex in found_examples:
    non_en_chars = sum(len(s) for s in ex["non_english_sentences"])
    ratio = non_en_chars / max(len(ex["full_text"]), 1)
    ratios.append(ratio)

if ratios:
    print(f"비영어 문자 비율        : 평균 {sum(ratios)/len(ratios)*100:.2f}%")

발견된 문서 수          : 1
문서당 비영어 문장 수   : 평균 1.00 | 최대 1
문서 수준 감지 언어 분포: {'en': 1}
비영어 문자 비율        : 평균 0.41%
